## JUMP Dataset Noise Model Creation for MicroSplit

In [10]:
# Import all the things we need further down
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import os

from careamics import CAREamist
from careamics.models.lvae.noise_models import GaussianMixtureNoiseModel, create_histogram
from careamics.lvae_training.dataset import DataSplitType
from careamics.config import GaussianMixtureNMConfig, create_n2v_configuration

In [11]:
# Define all available channels - move to dataset specific scripts
class Channels:
    DNA = "DNA"
    Mito = "Mito"
    RNA = "RNA"
    ER = "ER"
    AGP = "AGP"

# List of all channels
ALL_CHANNELS = [Channels.DNA, Channels.Mito, Channels.RNA, Channels.ER, Channels.AGP]

# Load data from your microsplit_dataset created in prepJUMP
def load_data(dataset_dir, channel_names):
    """Load images for each channel from the prepared dataset"""
    all_images = []
    
    for channel in channel_names:
        # Find the channel directory (case-insensitive)
        channel_dir = None
        for dir_name in os.listdir(dataset_dir):
            if dir_name.lower() == channel.lower() and os.path.isdir(os.path.join(dataset_dir, dir_name)):
                channel_dir = os.path.join(dataset_dir, dir_name)
                break
        
        if not channel_dir:
            print(f"Channel directory for '{channel}' not found in {dataset_dir}")
            continue
            
        # Load all tiff files for this channel
        files = sorted([f for f in os.listdir(channel_dir) if f.endswith('.tif')])
        
        # Load each image for this channel
        channel_images = []
        for f in files:
            img = tifffile.imread(os.path.join(channel_dir, f))
            channel_images.append(img)
        
        # Stack images for this channel
        if channel_images:
            channel_images = np.stack(channel_images)
            all_images.append(channel_images)
        else:
            print(f"No images found for channel {channel}")
    
    if not all_images:
        raise ValueError(f"No images loaded for any of the channels: {channel_names}")
        
    return np.stack(all_images, axis=-1)

### We need to create a noise model for each channel included in the dataset, so we need to process each channel individually 

In [12]:
# Process each channel individually
for channel in ALL_CHANNELS:
    print(f"\n\n{'='*50}")
    print(f"Processing channel: {channel}")
    print(f"{'='*50}")
    
    # Find datasets containing this channel
    dataset_dirs = []
    experiments_dir = "rand-3"  # Path to your experiments directory
    if os.path.exists(experiments_dir) and os.path.isdir(experiments_dir):
        # Search through combination sizes (2_channels, 3_channels, etc.)
        for size_dir in os.listdir(experiments_dir):
            size_path = os.path.join(experiments_dir, size_dir)
            if not os.path.isdir(size_path):
                continue
                
            # Search through combinations for this size
            for combo_dir in os.listdir(size_path):
                combo_path = os.path.join(size_path, combo_dir)
                if not os.path.isdir(combo_path):
                    continue
                    
                # Check if this combination contains our channel
                channel_dir = os.path.join(combo_path, channel)
                if os.path.isdir(channel_dir):
                    dataset_dirs.append(combo_path)
                    break  # Found one, no need to continue searching this size
    
    if not dataset_dirs:
        print(f"No dataset found containing {channel}")
        continue
        
    dataset_dir = dataset_dirs[0]  
    print(f"Using dataset: {dataset_dir}")
    
    try:
        input_data = load_data(dataset_dir, [channel])
        print(f"Input data shape: {input_data.shape}")
        
        # Train N2V 
        config = create_n2v_configuration(
            experiment_name=f"rand-3_noise_models_n2v_{channel}",
            data_type="array",
            axes="SYXC", 
            n_channels=1,  # Just one channel at a time
            patch_size=(64, 64),
            batch_size=64,
            num_epochs=10,
        )
        
        # Train N2V on the data
        careamist = CAREamist(source=config, work_dir=f"noise_models_{channel}")
        careamist.train(train_source=input_data, val_minimum_split=5)
        
        # Denoise data with the N2V model
        prediction = careamist.predict(input_data, tile_size=(256, 256))
        
        # Train the Noise Model for this channel
        print(f"Training noise model for channel {channel}")
        channel_data = input_data[..., 0]  # Since we're only loading one channel
        channel_prediction = np.concatenate(prediction)[:, 0]  # Get the denoised channel
        
        noise_model_config = GaussianMixtureNMConfig(
            model_type="GaussianMixtureNoiseModel",
            min_signal=channel_data.min(),
            max_signal=channel_data.max(),
            n_coeff=4,
            n_gaussian=6
        )
        
        noise_model = GaussianMixtureNoiseModel(noise_model_config)
        noise_model.fit(signal=channel_data, observation=channel_prediction, n_epochs=100)
        
        # Create noise_models directory if it doesn't exist and save 
        os.makedirs("noise_models", exist_ok=True)
        noise_model.save(path="noise_models", name=f"noise_model_{channel}")
        
        # Show the result
        histogram = create_histogram(
            bins=100,
            min_val=channel_data.min(),
            max_val=channel_data.max(),
            signal=channel_data,
            observation=channel_prediction
        )
        
        from microsplit_reproducibility.utils.utils import plot_probability_distribution
        plot_probability_distribution(
            noise_model,
            signalBinIndex=50,
            histogram=histogram[0],
            channel=0  # Since we're only using one channel
        )
        
    except Exception as e:
        print(f"Error processing channel {channel}: {str(e)}")



Processing channel: DNA
Using dataset: rand-3/5_channels/dna_rna_er_agp_mito
Input data shape: (100, 1080, 1280, 1)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Computed dataset mean: [151.53833303], std: [102.14422503]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type | Params | Mode 
---------------------------------------
0 | model | UNet | 509 K  | train
---------------------------------------
509 K     Trainable params
0         Non-trainable params
509 K     Total params
2.037     Total estimated model params size (MB)
39        Modules in train mode
0         Modules in eval mode


Epoch 9: 100%|██████████| 479/479 [00:16<00:00, 29.02it/s, train_loss_step=0.0583, val_loss=0.0237, train_loss_epoch=0.0273]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 479/479 [00:16<00:00, 28.79it/s, train_loss_step=0.0583, val_loss=0.0237, train_loss_epoch=0.0273]


...

Epoch 9: 100%|██████████| 479/479 [00:17<00:00, 27.96it/s, train_loss_step=0.0793, val_loss=0.0573, train_loss_epoch=0.0646]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 479/479 [00:17<00:00, 27.66it/s, train_loss_step=0.0793, val_loss=0.0573, train_loss_epoch=0.0646]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 3000/3000 [00:12<00:00, 233.34it/s]
Training noise model for channel AGP
[GaussianMixtureNoiseModel] min_sigma: 200.0
0 5.575198650360107

The trained parameters (noise_model_AGP) is saved at location: noise_models
Error processing channel AGP: No module named 'microsplit_reproducibility'
